# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohamedKroush/Flyrank-ML1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
#setup cell

%pip -q install duckdb huggingface_hub

import os, getpass, pandas as pd, numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:

    try:

        from google.colab import userdata

        HF_TOKEN = userdata.get('HF_TOKEN')

    except Exception:

        pass

HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb

con = duckdb.connect()

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"}

os.makedirs("work/outputs", exist_ok=True)

print("Ready.")

Ready.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
labeled = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
                    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last15,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev15,
               AVG(f.gsc_avg_position) AS avg_position_month,
               SUM(f.gsc_impressions) AS impressions_month,
               SUM(f.gsc_clicks) AS clicks_month
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
        GROUP BY 1,2
        HAVING imp_prev15 >= 50
    )
    SELECT *, (imp_last15 < 0.8 * imp_prev15)::INT AS is_declining
    FROM windowed
""").df()
print(f"{len(labeled):,} rows, decline base rate: {labeled['is_declining'].mean():.3f}")

# Signal check 1: impressions volume
labeled["impressions_bucket"] = pd.qcut(labeled["impressions_month"], 4, duplicates="drop")
bucket1 = labeled.groupby("impressions_bucket")["is_declining"].agg(["mean", "count"])
bucket1.columns = ["decline_rate", "n"]
print("Signal 1: impressions volume")
print(bucket1)

# Signal check 2: position
labeled["position_bucket"] = pd.cut(labeled["avg_position_month"], bins=[0,10,20,30,100], labels=["1-10","11-20","21-30","31+"])
bucket2 = labeled.groupby("position_bucket")["is_declining"].agg(["mean", "count"])
bucket2.columns = ["decline_rate", "n"]
print("Signal 2: position")
print(bucket2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

94,559 rows, decline base rate: 0.373
Signal 1: impressions volume
                    decline_rate      n
impressions_bucket                     
(49.999, 306.0]         0.486857  23662
(306.0, 859.0]          0.366691  23633
(859.0, 2728.5]         0.344819  23624
(2728.5, 617124.0]      0.294120  23640
Signal 2: position
                 decline_rate      n
position_bucket                     
1-10                 0.348230  52652
11-20                0.391492  20146
21-30                0.430083  10584
31+                  0.403543  11176


/tmp/ipykernel_1292/1042861140.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket1 = labeled.groupby("impressions_bucket")["is_declining"].agg(["mean", "count"])
/tmp/ipykernel_1292/1042861140.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket2 = labeled.groupby("position_bucket")["is_declining"].agg(["mean", "count"])


Rule (plain words): A page is worth reviewing if it's getting meaningful visibility (impressions ≥ 250) — since low-visibility pages decline much more often (48.7% at the bottom bucket vs. 29.4% at the top) — combined with a weak search position (>20), since decline rate roughly rises with worse position, from 34.8% at positions 1–10 up to ~40–43% beyond position 20.

Reason code: low_position_visible_page

Signal 1 verdict (impressions volume, behind quick-win logic): CONFIRMED — decline rate drops steadily from 48.7% (lowest-volume bucket, n=23,662) to 29.4% (highest-volume bucket, n=23,640). Low-visibility pages are the real risk group.

Signal 2 verdict (position, behind CTR-fix logic): MIXED — decline rate does rise from 34.8% (positions 1–10, n=52,652) to a peak of 43.0% (positions 21–30, n=10,584), but it's not perfectly monotonic (positions 31+ actually drop slightly to 40.4%, n=11,176). The overall direction supports the rule, but not cleanly.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Wrote 94559 rows
Precision@50: 0.480
Base rate: 0.373


,client_hash_id,content_hash_id,score,reason_code,action,impressions_month,avg_position_month,is_declining
13526,client_23a62021009f63c4,content_36e53e9c707674fc,194579.0,low_position_visible_page,review_for_ctr_or_content_fix,194579.0,32.766674,1
67973,client_20259bd6705d81d4,content_82e35c4845e6c391,143907.0,low_position_visible_page,review_for_ctr_or_content_fix,143907.0,22.558608,0
44964,client_23a62021009f63c4,content_3df3f32f3fd58dea,140156.0,low_position_visible_page,review_for_ctr_or_content_fix,140156.0,23.335465,1
91809,client_23a62021009f63c4,content_df47d1b976106de4,131707.0,low_position_visible_page,review_for_ctr_or_content_fix,131707.0,24.355625,0
44897,client_23a62021009f63c4,content_bdf60c86117079be,112429.0,low_position_visible_page,review_for_ctr_or_content_fix,112429.0,30.769353,1
63690,client_23a62021009f63c4,content_661a7734f691bef5,110424.0,low_position_visible_page,review_for_ctr_or_content_fix,110424.0,23.888656,0
15369,client_23a62021009f63c4,content_cae701a83cad5e36,98572.0,low_position_visible_page,review_for_ctr_or_content_fix,98572.0,23.730705,0
61387,client_23a62021009f63c4,content_559cdd76da9306de,97378.0,low_position_visible_page,review_for_ctr_or_content_fix,97378.0,36.712074,1
20741,client_20259bd6705d81d4,content_9fff53e827550f9d,94673.0,low_position_visible_page,review_for_ctr_or_content_fix,94673.0,22.469914,1
34775,client_fef1a8f436438636,content_ba462518dad435fc,91391.0,low_position_visible_page,review_for_ctr_or_content_fix,91391.0,27.355006,0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#Top-20 review (rule: low_position_visible_page, action: review_for_ctr_or_content_fix) content_36e53e9c... (194,579 impressions, position 32.8) — flagged for very high visibility at a weak position. Actually declining (✓). Would be wrong if this page recently improved position and the monthly average just hasn't caught up yet. content_82e35c48... (143,907 impr, position 22.6) — high visibility, borderline weak position. NOT declining (✗ — a miss). Would be wrong if position 22.6 isn't actually "weak" for this content type/intent — the threshold of 20 may be too aggressive here. content_3df3f32f... (140,156 impr, position 23.3) — similar profile to #2. Declining (✓). Reasonable flag given the pattern. content_df47d1b9... (131,707 impr, position 24.4) — high visibility, weak position. NOT declining (✗). Another miss near the position-20 threshold, suggesting the cutoff may be too coarse. content_bdf60c86... (112,429 impr, position 30.8) — high visibility, clearly weak position. Declining (✓). Strong candidate — matches the rule's intent well. content_661a7734... (110,424 impr, position 23.9) — NOT declining (✗). Same pattern as #2/#4 — position just past the threshold but page still stable. content_cae701a8... (98,572 impr, position 23.7) — NOT declining (✗). Consistent miss cluster around position ~22–24. content_559cdd76... (97,378 impr, position 36.7) — Declining (✓). Clearly weak position, good flag. content_9fff53e8... (94,673 impr, position 22.5) — Declining (✓). Borderline position but correctly flagged this time. content_ba462518... (91,391 impr, position 27.4) — NOT declining (✗). Would be wrong if this page has a stable, recurring position around 27 that isn't actually deteriorating. content_84a6bf35... (91,388 impr, position 20.8) — NOT declining (✗). Right at the position-20 boundary — a strong sign the threshold itself needs tuning. content_164c1f53... (89,982 impr, position 24.1) — NOT declining (✗). Same near-threshold miss pattern. content_b51957d7... (85,219 impr, position 28.8) — NOT declining (✗). Would be wrong if this page is a long-term stable page that just naturally ranks around position 29. content_6486239... (83,293 impr, position 29.1) — Declining (✓). Good flag. content_89c10d52... (81,777 impr, position 22.4) — NOT declining (✗). Another near-boundary miss. content_9d1e94cb... (81,479 impr, position 34.6) — Declining (✓). Clearly weak position, strong flag. content_73aa61dc... (80,124 impr, position 45.7) — NOT declining (✗) — surprising, since this is the worst position in the top 20. Would be wrong if this page is a known low-priority/legacy page that isn't expected to perform regardless. content_fa84f597... (78,716 impr, position 36.5) — NOT declining (✗). Similar surprise — deep position but stable. content_b88025fb... (78,328 impr, position 25.2) — Declining (✓). Reasonable flag. content_ab91e088... (77,963 impr, position 43.2) — NOT declining (✗). Another deep-position page that's stable, not declining.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#Weak picks: Rows #17 and #20 stand out as weak — both sit at very deep positions (45.7 and 43.2) yet are NOT declining, meaning the rule's core assumption ("worse position → more likely declining") breaks down at the extreme tail. This suggests some deep-position pages are simply stable/steady-state rather than actively worsening, and the rule can't distinguish "consistently poor" from "actively declining."
#Leakage check: No product-decision flags (health_score, priority_score, action_type) were used anywhere — only impressions_month and avg_position_month, both computed purely from the observed 2026-03 window. The is_declining label (based on a 15-day split) is used only to evaluate the rule's precision, never as an input to the score itself. No future-window data leaked into the scoring formula.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.